# TinyVLM with BitNet b1.58  (W1.58A8)


## 1. Instalacion

In [ ]:
# ==============================================================================
# 1  INSTALLATION
# ==============================================================================
import subprocess, sys

def _pip(*p): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *p])

_pip("transformers>=4.40", "tokenizers>=0.15", "Pillow", "tqdm", "matplotlib", "psutil", "ninja")

print("Dependencies installed")



Dependencies installed


## 2. Imports and confg

In [ ]:
# ==============================================================================
# 2  IMPORTS & SEEDS
# ==============================================================================

import os, math, json, random, time, zipfile, textwrap, gc
import urllib.request
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from PIL import Image
import torchvision.transforms as T
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as BLD
from transformers import CLIPModel
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import psutil
from tqdm import tqdm

def set_seed(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE  = torch.float16 if device.type == "cuda" else torch.float32
if device.type == "cuda":
    p = torch.cuda.get_device_properties(0)
    print(f"{p.name}  ({p.total_memory/1e9:.1f} GB)")


# ==============================================================================
# 3  CONFIGURATION
# ==============================================================================


# -- CLIP (frozen encoder) -----------------------------------------------------
CLIP_ID   = "openai/clip-vit-base-patch16"
CLIP_DIM  = 768     # ViT-B/16 output dimension
N_PATCHES = 196     # (224/16)^2

# -- Language model ------------------------------------------------------------
class LMCfg:
    vocab_size  = 8_192   # adjusted after BPE training
    max_seq_len = 64
    dim         = 1024
    depth       = 12
    num_heads   = 16       # 384/6 = 64 per head
    mlp_ratio   = 4.0
    dropout     = 0.1
    bitnet      = True    # use BitLinear in TinyGPT projections
    bitnet_act_bits = 8   # W1.58A8: ternary weights, int8 activations
    bitnet_cuda_kernel = True  # optional CUDA kernel for eval/inference
    bitnet_compile_cuda = True # falls back to PyTorch if compilation fails

# -- Training ------------------------------------------------------------------
class TrainCfg:
    batch_size      = 64   # large: batches are cached tensors only
    grad_accum      = 1    # effective batch = 64 (no accumulation needed)
    lr              = 3e-4
    min_lr          = 3e-5
    weight_decay    = 0.05
    beta1, beta2    = 0.9, 0.95
    grad_clip       = 1.0
    warmup_steps    = 300
    max_steps       = 8_000
    log_every       = 20
    eval_every      = 400
    save_every      = 2_000
    mixed_precision = (device.type == "cuda")
    root        = Path("/content/tinyvlm3")
    coco_dir    = Path("/content/coco")

lcfg = LMCfg()
tcfg = TrainCfg()
tcfg.root.mkdir(parents=True, exist_ok=True)
(tcfg.root/"ckpt").mkdir(exist_ok=True)

print("Config ready")
print(f"  batch_size : {tcfg.batch_size}")
print(f"  max_steps  : {tcfg.max_steps:,}")

Tesla T4  (15.6 GB)
Config ready
  batch_size : 64
  max_steps  : 8,000


## 3.  COCO val2017


In [ ]:
# ==============================================================================
# 4  COCO val2017 DOWNLOAD
# ==============================================================================


URLS = {
    "val2017.zip":
        "http://images.cocodataset.org/zips/val2017.zip",
    "annotations_trainval2017.zip":
        "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
}

def _hook(c, bs, tot):
    pct = min(c*bs*100/tot, 100)
    print(f"\r  [{'#'*int(pct/2)+'-'*(50-int(pct/2))}] {pct:5.1f}%", end="", flush=True)

def download_coco():
    tcfg.coco_dir.mkdir(parents=True, exist_ok=True)
    for fname, url in URLS.items():
        fpath  = tcfg.coco_dir / fname
        marker = tcfg.coco_dir / fname.replace(".zip","_ok")
        if not fpath.exists():
            print(f"Downloading {fname}")
            urllib.request.urlretrieve(url, fpath, _hook); print()
        if not marker.exists():
            print(f"Extracting {fname} ...")
            with zipfile.ZipFile(fpath) as z: z.extractall(tcfg.coco_dir)
            marker.touch()
        else:
            print(f"Already downloaded: {fname}")

download_coco()

ann = tcfg.coco_dir/"annotations"/"captions_val2017.json"
with open(ann) as f: raw = json.load(f)

id2file = {i["id"]: i["file_name"] for i in raw["images"]}
id2caps: Dict[int,List[str]] = {}
for a in raw["annotations"]:
    id2caps.setdefault(a["image_id"],[]).append(a["caption"])

# one (image, caption) pair per annotation entry
pairs: List[Tuple[str,str]] = []
for iid, caps in id2caps.items():
    p = tcfg.coco_dir/"val2017"/id2file[iid]
    if p.exists():
        for c in caps:
            pairs.append((str(p), c.strip()))

random.shuffle(pairs)
split       = int(0.9*len(pairs))
train_pairs = pairs[:split]
val_pairs   = pairs[split:]

print(f"Pairs: {len(pairs):,}  (train={len(train_pairs):,}  val={len(val_pairs):,})")

  [##################################################] 100.0%
Extracting val2017.zip ...
  [##################################################] 100.0%
Extracting annotations_trainval2017.zip ...
Pairs: 25,014  (train=22,512  val=2,502)


## 4. Tokenizer BPE


In [ ]:
# ==============================================================================
# 5  BPE TOKENIZER FROM SCRATCH
# ==============================================================================

SPECIAL = ["[PAD]","[UNK]","[BOS]","[EOS]"]
TOK_PATH = tcfg.root/"tokenizer.json"

def train_tok(caps, vocab_size):
    tmp = Path("/tmp/caps.txt")
    tmp.write_text("\n".join(c.lower() for c in caps))
    tok     = Tokenizer(BPE(unk_token="[UNK]"))
    tok.pre_tokenizer = ByteLevel(add_prefix_space=False)
    tok.decoder       = BLD()
    trainer = BpeTrainer(vocab_size=vocab_size, special_tokens=SPECIAL,
                         min_frequency=2, show_progress=True)
    tok.train([str(tmp)], trainer)
    return tok

if TOK_PATH.exists():
    tokenizer = Tokenizer.from_file(str(TOK_PATH))
    print(f"Tokenizer loaded (vocab={tokenizer.get_vocab_size()})")
else:
    print("Training BPE tokenizer ...")
    tokenizer = train_tok([c for _,c in pairs], lcfg.vocab_size)
    tokenizer.save(str(TOK_PATH))
    print(f"Tokenizer saved (vocab={tokenizer.get_vocab_size()})")

lcfg.vocab_size = tokenizer.get_vocab_size()
PAD_ID = tokenizer.token_to_id("[PAD]")
BOS_ID = tokenizer.token_to_id("[BOS]")
EOS_ID = tokenizer.token_to_id("[EOS]")

def encode(text, max_len=lcfg.max_seq_len):
    ids = [BOS_ID] + tokenizer.encode(text.lower()).ids[:max_len-2] + [EOS_ID]
    return ids

def decode(ids):
    return tokenizer.decode([i for i in ids if i not in (BOS_ID,EOS_ID,PAD_ID)])

# BPE may alter spacing on certain tokens, so no hard assert here
t = "a cat sitting on a couch"
print(f"Tokenizer OK")
print(f"  original : '{t}'")
print(f"  decoded  : '{decode(encode(t))}'")

Training BPE tokenizer ...
Tokenizer saved (vocab=7626)
Tokenizer OK
  original : 'a cat sitting on a couch'
  decoded  : 'a cat sitting on a couch'


## 5. Cache features CLIP


In [ ]:
# ==============================================================================
# 6  CLIP FEATURE CACHING
#    Runs once and saves one tensor per image to disk.
#    The training loop never touches CLIP again — only reads cached tensors.
#    This makes training 5-8x faster.
# ==============================================================================

CLIP_MEAN  = [0.48145466, 0.4578275,  0.40821073]
CLIP_STD   = [0.26862954, 0.26130258, 0.27577711]
clip_tf    = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(CLIP_MEAN, CLIP_STD),
])

# single file ~750 MB fp16 (vs ~3 GB with individual files)
CACHE_FILE = tcfg.root / "clip_features.pt"

# remove cache folders from previous runs to free disk space
import shutil
for old in ["/content/tinyvlm_ckpt", "/content/tinyvlm2_ckpt",
            "/content/tinyvlm3/clip_cache"]:
    if Path(old).exists():
        shutil.rmtree(old, ignore_errors=True)
        print(f"Removed old cache folder: {old}")

# global dict: stem -> fp32 tensor in RAM (~750 MB fp16 -> 1.5 GB fp32)
feat_cache: Dict[str, torch.Tensor] = {}


def build_cache():
    global feat_cache

    unique_paths = list({p for p, _ in pairs})

    # if the cache file already exists, load it into RAM
    if CACHE_FILE.exists():
        print("Loading features into RAM ...")
        saved = torch.load(CACHE_FILE, map_location="cpu")
        feat_cache = {k: v.to(torch.float32) for k, v in saved.items()}
        print(f"Cache loaded — {len(feat_cache):,} images in RAM")
        return

    # first run: extract with CLIP and save everything to a single file
    print("Extracting CLIP features (runs once, ~3-5 min) ...")
    _clip = CLIPModel.from_pretrained(CLIP_ID).vision_model.to(device).eval()
    for p in _clip.parameters():
        p.requires_grad_(False)

    bs     = 64
    result = {}   # stem -> fp16 tensor

    with torch.no_grad():
        for i in tqdm(range(0, len(unique_paths), bs), desc="  CLIP encode"):
            batch = unique_paths[i:i+bs]
            imgs, stems = [], []
            for pp in batch:
                try:
                    imgs.append(clip_tf(Image.open(pp).convert("RGB")))
                    stems.append(Path(pp).stem)
                except Exception:
                    continue
            if not imgs:
                continue
            tensor = torch.stack(imgs).to(device)
            with torch.cuda.amp.autocast(enabled=True, dtype=torch.float16):
                feats = _clip(pixel_values=tensor).last_hidden_state[:, 1:, :]
            feats = feats.cpu().to(torch.float16)   # (B, 196, 768) fp16
            for stem, feat in zip(stems, feats):
                result[stem] = feat

    del _clip
    torch.cuda.empty_cache()

    print(f"Saving {len(result):,} features to {CACHE_FILE} ...")
    torch.save(result, CACHE_FILE)
    sz = CACHE_FILE.stat().st_size / 1e6
    print(f"Saved — {sz:.0f} MB  (single file)")

    # load into RAM as fp32
    feat_cache = {k: v.to(torch.float32) for k, v in result.items()}
    del result


build_cache()


def load_feat(img_path: str) -> Optional[torch.Tensor]:
    """Returns the (196, 768) fp32 feature from RAM. None if not found."""
    return feat_cache.get(Path(img_path).stem)


# sanity check
_f = load_feat(pairs[0][0])
assert _f is not None and _f.shape == (196, 768), f"Cache fail: shape={getattr(_f,'shape',None)}"
print(f"Cache verified — {len(feat_cache):,} images  shape={_f.shape}  dtype={_f.dtype}")
del _f

Extracting CLIP features (runs once, ~3-5 min) ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/599M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch16
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.

  CLIP encode:   0%|          | 0/79 [00:00<?, ?it/s]/tmp/ipykernel_17639/2732441459.py:66: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=True, dtype=torch.float16):

  CLIP encode: 100%|██████████| 79/79 [01:00<00:00,  1.30it/s]


Saving 5,000 features to /content/tinyvlm3/clip_features.pt ...
Saved — 1506 MB  (single file)
Cache verified — 5,000 images  shape=torch.Size([196, 768])  dtype=torch.float32


## 6. Architecture
- **Projector**: nn.Linear fp32, CLIP (768) -> GPT dim (384)
- **BitLinear**: weights fp32 shadow, STE ternario training.
- **TinyGPT**: 6 bloks, cross-attention
- **TinyVLM**: Projector and TinyGPT

In [ ]:
# ==============================================================================
# 7  PROJECTOR  768 -> 384
# ==============================================================================

class Projector(nn.Module):
    def __init__(self, in_d, out_d):
        super().__init__()
        mid = out_d * 2
        self.fc1  = nn.Linear(in_d, mid)
        self.fc2  = nn.Linear(mid, out_d)
        self.skip = nn.Linear(in_d, out_d, bias=False)
        self.norm = nn.LayerNorm(out_d)
        self.act  = nn.GELU()
        for m in [self.fc1, self.fc2, self.skip]:
            nn.init.xavier_uniform_(m.weight)

    def forward(self, x):
        return self.norm(self.fc2(self.act(self.fc1(x))) + self.skip(x))


# ==============================================================================
# 8  TINYGPT WITH CROSS-ATTENTION
# ==============================================================================

# BitNet b1.58 / W1.58A8 for TinyGPT.
# BitLinear ternarizes weights {-1, 0, +1} via absmean and quantizes
# activations to int8 via absmax. During training, shadow weights are kept
# in full precision with STE; savings materialize when exporting packed weights.
BITNET_CUDA_EXT = None

def rss_gb():
    try:
        return psutil.Process(os.getpid()).memory_info().rss / (1024**3)
    except Exception:
        return float('nan')

def cuda_mem_gb(kind='allocated'):
    if device.type != 'cuda': return 0.0
    fn = torch.cuda.memory_allocated if kind == 'allocated' else torch.cuda.memory_reserved
    return fn() / (1024**3)

def cuda_peak_gb():
    if device.type != 'cuda': return 0.0
    return torch.cuda.max_memory_allocated() / (1024**3)

def _build_bitnet_cuda_ext(enabled=True):
    if not enabled or device.type != 'cuda':
        return None
    try:
        from torch.utils.cpp_extension import load_inline
        cpp_src = r'''
#include <torch/extension.h>
torch::Tensor bitlinear_forward_cuda(torch::Tensor x_q, torch::Tensor w_q,
                                     torch::Tensor x_scale, torch::Tensor w_scale);
torch::Tensor bitlinear_packed_forward_cuda(torch::Tensor x_q, torch::Tensor w_packed,
                                            torch::Tensor x_scale, torch::Tensor w_scale,
                                            int64_t K);
torch::Tensor bitlinear_forward(torch::Tensor x_q, torch::Tensor w_q,
                                torch::Tensor x_scale, torch::Tensor w_scale) {
    TORCH_CHECK(x_q.is_cuda(), "x_q must be CUDA");
    TORCH_CHECK(w_q.is_cuda(), "w_q must be CUDA");
    TORCH_CHECK(x_q.dtype() == torch::kInt8, "x_q must be int8");
    TORCH_CHECK(w_q.dtype() == torch::kInt8, "w_q must be int8");
    TORCH_CHECK(x_q.dim() == 2 && w_q.dim() == 2, "expected 2D matrices");
    TORCH_CHECK(x_q.size(1) == w_q.size(1), "K mismatch");
    return bitlinear_forward_cuda(x_q.contiguous(), w_q.contiguous(),
                                  x_scale.contiguous(), w_scale.contiguous());
}
torch::Tensor bitlinear_packed_forward(torch::Tensor x_q, torch::Tensor w_packed,
                                       torch::Tensor x_scale, torch::Tensor w_scale,
                                       int64_t K) {
    TORCH_CHECK(x_q.is_cuda(), "x_q must be CUDA");
    TORCH_CHECK(w_packed.is_cuda(), "w_packed must be CUDA");
    TORCH_CHECK(x_q.dtype() == torch::kInt8, "x_q must be int8");
    TORCH_CHECK(w_packed.dtype() == torch::kUInt8, "w_packed must be uint8");
    TORCH_CHECK(x_q.dim() == 2 && w_packed.dim() == 2, "expected 2D matrices");
    TORCH_CHECK(x_q.size(1) == K, "K mismatch");
    TORCH_CHECK(w_packed.size(1) == (K + 3) / 4, "packed K mismatch");
    return bitlinear_packed_forward_cuda(x_q.contiguous(), w_packed.contiguous(),
                                         x_scale.contiguous(), w_scale.contiguous(), K);
}
'''
        cuda_src = r'''
#include <torch/extension.h>
#include <c10/cuda/CUDAException.h>
#include <cuda.h>
#include <cuda_runtime.h>
#include <cstdint>

__global__ void bitlinear_kernel(const int8_t* __restrict__ x,
                                 const int8_t* __restrict__ w,
                                 const float* __restrict__ xs,
                                 const float* __restrict__ ws,
                                 float* __restrict__ y,
                                 int M, int K, int N) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    if (row >= M || col >= N) return;
    int acc = 0;
    const int8_t* xr = x + row * K;
    const int8_t* wr = w + col * K;
    for (int k = 0; k < K; ++k) {
        acc += static_cast<int>(xr[k]) * static_cast<int>(wr[k]);
    }
    y[row * N + col] = static_cast<float>(acc) * xs[row] * ws[0];
}

__global__ void bitlinear_packed_kernel(const int8_t* __restrict__ x,
                                        const uint8_t* __restrict__ w,
                                        const float* __restrict__ xs,
                                        const float* __restrict__ ws,
                                        float* __restrict__ y,
                                        int M, int K, int packed_K, int N) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    if (row >= M || col >= N) return;
    int acc = 0;
    const int8_t* xr = x + row * K;
    const uint8_t* wr = w + col * packed_K;
    for (int pk = 0; pk < packed_K; ++pk) {
        uint8_t byte = wr[pk];
        #pragma unroll
        for (int j = 0; j < 4; ++j) {
            int k = pk * 4 + j;
            if (k >= K) break;
            int code = (byte >> (2 * j)) & 0x03;
            int wv = code - 1;  // 0,1,2 -> -1,0,+1
            acc += static_cast<int>(xr[k]) * wv;
        }
    }
    y[row * N + col] = static_cast<float>(acc) * xs[row] * ws[0];
}

torch::Tensor bitlinear_forward_cuda(torch::Tensor x_q, torch::Tensor w_q,
                                     torch::Tensor x_scale, torch::Tensor w_scale) {
    const int M = static_cast<int>(x_q.size(0));
    const int K = static_cast<int>(x_q.size(1));
    const int N = static_cast<int>(w_q.size(0));
    auto y = torch::empty({M, N}, x_q.options().dtype(torch::kFloat32));
    dim3 block(16, 16);
    dim3 grid((N + block.x - 1) / block.x, (M + block.y - 1) / block.y);
    bitlinear_kernel<<<grid, block>>>(
        reinterpret_cast<const int8_t*>(x_q.data_ptr<int8_t>()),
        reinterpret_cast<const int8_t*>(w_q.data_ptr<int8_t>()),
        x_scale.data_ptr<float>(), w_scale.data_ptr<float>(),
        y.data_ptr<float>(), M, K, N);
    C10_CUDA_KERNEL_LAUNCH_CHECK();
    return y;
}

torch::Tensor bitlinear_packed_forward_cuda(torch::Tensor x_q, torch::Tensor w_packed,
                                            torch::Tensor x_scale, torch::Tensor w_scale,
                                            int64_t K64) {
    const int M = static_cast<int>(x_q.size(0));
    const int K = static_cast<int>(K64);
    const int packed_K = static_cast<int>(w_packed.size(1));
    const int N = static_cast<int>(w_packed.size(0));
    auto y = torch::empty({M, N}, x_q.options().dtype(torch::kFloat32));
    dim3 block(16, 16);
    dim3 grid((N + block.x - 1) / block.x, (M + block.y - 1) / block.y);
    bitlinear_packed_kernel<<<grid, block>>>(
        reinterpret_cast<const int8_t*>(x_q.data_ptr<int8_t>()),
        reinterpret_cast<const uint8_t*>(w_packed.data_ptr<uint8_t>()),
        x_scale.data_ptr<float>(), w_scale.data_ptr<float>(),
        y.data_ptr<float>(), M, K, packed_K, N);
    C10_CUDA_KERNEL_LAUNCH_CHECK();
    return y;
}
'''
        ext = load_inline(
            name='tinyvlm_bitlinear_cuda_ext',
            cpp_sources=cpp_src,
            cuda_sources=cuda_src,
            functions=['bitlinear_forward', 'bitlinear_packed_forward'],
            extra_cuda_cflags=['-O3'],
            verbose=False,
        )
        print('  BitNet CUDA kernel compiled')
        return ext
    except Exception as e:
        print(f'  BitNet CUDA kernel unavailable ({type(e).__name__}: {e}); falling back to PyTorch')
        return None

def _ste_quantize(x, dequant):
    return x + (dequant - x).detach()

def _ternary_dequant_weight(w, eps=1e-5):
    scale = w.detach().abs().mean().clamp_min(eps)
    q = torch.clamp(torch.round(w / scale), -1, 1)
    return _ste_quantize(w, q * scale)

def _absmax_dequant_act(x, bits=8, eps=1e-5):
    qmax = (2 ** (bits - 1)) - 1
    qmin = -2 ** (bits - 1)
    scale = x.detach().abs().amax(dim=-1, keepdim=True).clamp_min(eps) / qmax
    q = torch.clamp(torch.round(x / scale), qmin, qmax)
    return _ste_quantize(x, q * scale)

class BitLinear(nn.Module):
    def __init__(self, in_features, out_features, bias=False, act_bits=8, use_cuda_kernel=True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.act_bits = act_bits
        self.use_cuda_kernel = use_cuda_kernel
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias = nn.Parameter(torch.empty(out_features)) if bias else None
        self._packed_weight = None
        self._packed_scale = None
        self._packed_pad = 0
        self._packed_cache_key = None
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.normal_(self.weight, std=0.02)
        if self.bias is not None: nn.init.zeros_(self.bias)

    @torch.no_grad()
    def quantized_weight_int8(self):
        scale = self.weight.detach().abs().mean().clamp_min(1e-5)
        q = torch.clamp(torch.round(self.weight.detach() / scale), -1, 1).to(torch.int8)
        return q.contiguous(), scale.reshape(1).float().contiguous()

    @torch.no_grad()
    def packed_weight_2bit(self):
        key = (self.weight.data_ptr(), int(self.weight._version), str(self.weight.device))
        if self._packed_weight is None or self._packed_cache_key != key:
            q, scale = self.quantized_weight_int8()
            packed, pad = pack_ternary_2bit(q)
            self._packed_weight = packed.to(self.weight.device, non_blocking=True).contiguous()
            self._packed_scale = scale.to(self.weight.device, non_blocking=True).float().contiguous()
            self._packed_pad = pad
            self._packed_cache_key = key
        return self._packed_weight, self._packed_scale, self._packed_pad

    def _cuda_eval_forward(self, x):
        global BITNET_CUDA_EXT
        if BITNET_CUDA_EXT is None or not self.use_cuda_kernel or self.training or x.requires_grad or not x.is_cuda:
            return None
        shape = x.shape[:-1]
        x2 = x.reshape(-1, self.in_features).contiguous()
        qmax = (2 ** (self.act_bits - 1)) - 1
        qmin = -2 ** (self.act_bits - 1)
        xs = x2.detach().abs().amax(dim=-1, keepdim=True).clamp_min(1e-5) / qmax
        xq = torch.clamp(torch.round(x2 / xs), qmin, qmax).to(torch.int8).contiguous()
        wp, ws, _ = self.packed_weight_2bit()
        y = BITNET_CUDA_EXT.bitlinear_packed_forward(xq, wp, xs.squeeze(-1).float().contiguous(), ws, self.in_features)
        if self.bias is not None: y = y + self.bias.float()
        return y.to(dtype=x.dtype).reshape(*shape, self.out_features)

    def forward(self, x):
        y = self._cuda_eval_forward(x)
        if y is not None: return y
        xq = _absmax_dequant_act(x, self.act_bits)
        wq = _ternary_dequant_weight(self.weight)
        return F.linear(xq, wq, self.bias)

def make_gpt_linear(cfg, in_d, out_d, bias=False):
    if getattr(cfg, 'bitnet', False):
        return BitLinear(in_d, out_d, bias=bias,
                         act_bits=getattr(cfg, 'bitnet_act_bits', 8),
                         use_cuda_kernel=getattr(cfg, 'bitnet_cuda_kernel', True))
    return nn.Linear(in_d, out_d, bias=bias)

def pack_ternary_2bit(q):
    codes = (q.detach().to(torch.int16) + 1).to(torch.uint8).cpu()
    orig_shape = tuple(codes.shape)
    assert codes.dim() == 2, 'pack_ternary_2bit expects a 2D matrix'
    pad = (-codes.shape[1]) % 4
    if pad:
        codes = torch.cat([codes, torch.ones(codes.shape[0], pad, dtype=torch.uint8)], dim=1)
    v = codes.reshape(orig_shape[0], -1, 4).to(torch.int16)
    packed = (v[:,:,0] | (v[:,:,1] << 2) | (v[:,:,2] << 4) | (v[:,:,3] << 6)).to(torch.uint8)
    return packed.contiguous(), pad

def unpack_ternary_2bit(packed, shape, pad=0, device=None):
    if isinstance(shape, torch.Tensor):
        shape = tuple(int(v) for v in shape.tolist())
    b = packed.cpu().to(torch.int16)
    if b.dim() == 1:
        b = b.view(int(shape[0]), -1)
    vals = torch.empty(b.shape[0], b.shape[1] * 4, dtype=torch.int16)
    vals[:,0::4] = b & 0x03
    vals[:,1::4] = (b >> 2) & 0x03
    vals[:,2::4] = (b >> 4) & 0x03
    vals[:,3::4] = (b >> 6) & 0x03
    if pad:
        vals = vals[:,:-int(pad)]
    q = vals.sub(1).to(torch.int8).reshape(shape)
    return q.to(device) if device is not None else q

def bitnet_memory_report(model):
    bitmods = [(n,m) for n,m in model.named_modules() if isinstance(m, BitLinear)]
    shadow = sum(p.numel() * p.element_size() for p in model.parameters())
    bit_params = sum(m.weight.numel() for _,m in bitmods)
    packed2 = sum(m.out_features * ((m.in_features + 3)//4) + 4 for _,m in bitmods)
    entropy = sum(m.weight.numel() * math.log2(3) / 8 + 4 for _,m in bitmods)
    neg = zero = pos = 0
    for _,m in bitmods:
        q,_ = m.quantized_weight_int8()
        neg += int((q == -1).sum().item()); zero += int((q == 0).sum().item()); pos += int((q == 1).sum().item())
    print('\nBitNet report')
    print(f'  BitLinear modules       : {len(bitmods)}')
    print(f'  BitLinear weights       : {bit_params/1e6:.2f} M')
    if bit_params:
        print(f'  Ternary -1/0/+1        : {100*neg/bit_params:.1f}%/{100*zero/bit_params:.1f}%/{100*pos/bit_params:.1f}%')
    print(f'  Shadow model            : {shadow/1024**2:.1f} MiB')
    print(f'  GPT BitLinear packed 2b : {packed2/1024**2:.2f} MiB')
    print(f'  GPT theoretical 1.58b   : {entropy/1024**2:.2f} MiB')
    print(f'  Process RAM             : {rss_gb():.2f} GiB')
    if device.type == 'cuda':
        print(f'  VRAM alloc/reserved     : {cuda_mem_gb("allocated"):.2f}/{cuda_mem_gb("reserved"):.2f} GiB')
    return {'bitlinear_modules':len(bitmods), 'bitlinear_weights':bit_params,
            'shadow_bytes':shadow, 'packed2_bytes':packed2, 'theoretical_158_bytes':entropy}

def export_bitnet_packed_gpt(model, path=None):
    packed = {}
    for name, m in model.lm.named_modules():
        if isinstance(m, BitLinear):
            q, scale = m.quantized_weight_int8()
            p, pad = pack_ternary_2bit(q)
            packed[f'{name}.weight_packed_2bit'] = p
            packed[f'{name}.weight_shape'] = torch.tensor(q.shape, dtype=torch.int32)
            packed[f'{name}.weight_pad'] = torch.tensor([pad], dtype=torch.int32)
            packed[f'{name}.weight_scale'] = scale.cpu()
    if path is not None:
        torch.save(packed, path)
        print(f'  BitNet GPT packed weights saved to {path}')
    return packed

def profile_forward_once(model, feat, inp, lbl=None, repeats=8, warmup=2):
    was_training = model.training
    model.eval(); gc.collect()
    if device.type == 'cuda': torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    times = []
    with torch.no_grad():
        for i in range(warmup + repeats):
            if device.type == 'cuda': torch.cuda.synchronize()
            t = time.perf_counter()
            out = model(feat, inp, lbl) if lbl is not None else model.lm(inp, model.encode(feat))
            if device.type == 'cuda': torch.cuda.synchronize()
            if i >= warmup: times.append(time.perf_counter() - t)
    if was_training: model.train()
    avg = sum(times) / max(len(times), 1)
    print(f'  Forward benchmark: {avg*1000:.2f} ms  RAM={rss_gb():.2f} GiB  peakVRAM={cuda_peak_gb():.2f} GiB')
    return {'forward_ms':avg*1000, 'rss_gb':rss_gb(), 'peak_vram_gb':cuda_peak_gb()}

def set_bitnet_cuda_kernel(model, enabled=True):
    old = []
    for m in model.modules():
        if isinstance(m, BitLinear):
            old.append((m, m.use_cuda_kernel))
            m.use_cuda_kernel = enabled
    return old

def restore_bitnet_cuda_kernel(old):
    for m, enabled in old:
        m.use_cuda_kernel = enabled

def profile_bitnet_paths(model, feat, inp, lbl=None, repeats=4, warmup=1):
    old = set_bitnet_cuda_kernel(model, False)
    torch_path = profile_forward_once(model, feat, inp, lbl, repeats=repeats, warmup=warmup)
    set_bitnet_cuda_kernel(model, True)
    cuda_path = profile_forward_once(model, feat, inp, lbl, repeats=repeats, warmup=warmup)
    restore_bitnet_cuda_kernel(old)
    speedup = torch_path['forward_ms'] / max(cuda_path['forward_ms'], 1e-9)
    print(f'  BitNet CUDA/PyTorch speedup: {speedup:.2f}x')
    return {'torch_path':torch_path, 'cuda_path':cuda_path, 'speedup':speedup}

def audit_bitnet_tinygpt(model):
    bitmods = [(n,m) for n,m in model.lm.named_modules() if isinstance(m, BitLinear)]
    bad_linear = [(n,type(m).__name__) for n,m in model.lm.named_modules()
                  if isinstance(m, nn.Linear) and n != 'head']
    expected = len(model.lm.blocks) * 8
    assert not bad_linear, f'GPT Linear layers without BitLinear: {bad_linear}'
    assert len(bitmods) == expected, f'BitLinear expected={expected}, found={len(bitmods)}'
    assert model.lm.head.weight is model.lm.emb.weight, 'head must remain tied to embedding'
    print(f'  BitNet GPT audit OK: {len(bitmods)} BitLinear modules; head tied to embedding')
    return {'bitlinear_modules':len(bitmods), 'expected':expected, 'bad_linear':bad_linear}

def check_bitnet_pack_roundtrip(model, max_modules=None):
    checked = 0
    for name, m in model.lm.named_modules():
        if not isinstance(m, BitLinear):
            continue
        q, _ = m.quantized_weight_int8()
        p, pad = pack_ternary_2bit(q)
        rt = unpack_ternary_2bit(p, q.shape, pad=pad)
        assert torch.equal(q.cpu(), rt), f'pack/unpack mismatch in {name}'
        checked += 1
        if max_modules is not None and checked >= max_modules:
            break
    print(f'  2-bit pack roundtrip OK: {checked} modules')
    return checked

def check_bitnet_kernel_equivalence(model, max_modules=4, atol=2e-2):
    if device.type != 'cuda' or BITNET_CUDA_EXT is None:
        print('  CUDA equivalence: skipped (kernel not available)')
        return None
    was_training = model.training
    model.eval()
    checked, max_err = 0, 0.0
    for name, m in model.lm.named_modules():
        if not isinstance(m, BitLinear):
            continue
        x = torch.randn(3, 5, m.in_features, device=device, dtype=torch.float32)
        old = m.use_cuda_kernel
        m.use_cuda_kernel = False
        y_ref = m(x)
        m.use_cuda_kernel = True
        y_cuda = m(x)
        m.use_cuda_kernel = old
        err = (y_ref - y_cuda).abs().max().item()
        max_err = max(max_err, err)
        assert err <= atol, f'CUDA kernel mismatch in {name}: max_err={err:.4g}'
        checked += 1
        if checked >= max_modules:
            break
    if was_training:
        model.train()
    print(f'  CUDA equivalence OK: {checked} modules, max_err={max_err:.3g}')
    return {'checked':checked, 'max_err':max_err}

BITNET_CUDA_EXT = _build_bitnet_cuda_ext(getattr(lcfg, 'bitnet', False) and getattr(lcfg, 'bitnet_compile_cuda', True))

class CausalSA(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.h = cfg.num_heads; self.d = cfg.dim // cfg.num_heads
        self.qkv  = make_gpt_linear(cfg, cfg.dim, 3*cfg.dim, bias=False)
        self.proj = make_gpt_linear(cfg, cfg.dim, cfg.dim, bias=False)
        self.drop = cfg.dropout

    def forward(self, x):
        B,T,D = x.shape
        q,k,v = self.qkv(x).reshape(B,T,3,self.h,self.d).permute(2,0,3,1,4).unbind(0)
        o = F.scaled_dot_product_attention(q,k,v, is_causal=True,
                dropout_p=self.drop if self.training else 0.0)
        return self.proj(o.transpose(1,2).reshape(B,T,D))


class CrossAttn(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.h = cfg.num_heads; self.d = cfg.dim // cfg.num_heads
        self.q    = make_gpt_linear(cfg, cfg.dim, cfg.dim, bias=False)
        self.kv   = make_gpt_linear(cfg, cfg.dim, 2*cfg.dim, bias=False)
        self.proj = make_gpt_linear(cfg, cfg.dim, cfg.dim, bias=False)

    def forward(self, x, ctx):
        B,T,D = x.shape; S = ctx.size(1)
        q       = self.q(x).reshape(B,T,self.h,self.d).transpose(1,2)
        k,v     = self.kv(ctx).reshape(B,S,2,self.h,self.d).permute(2,0,3,1,4).unbind(0)
        o       = F.scaled_dot_product_attention(q,k,v)
        return self.proj(o.transpose(1,2).reshape(B,T,D))


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        h = int(cfg.dim * cfg.mlp_ratio)
        self.g = make_gpt_linear(cfg, cfg.dim, h, bias=False)
        self.u = make_gpt_linear(cfg, cfg.dim, h, bias=False)
        self.d = make_gpt_linear(cfg, h, cfg.dim, bias=False)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x):
        return self.drop(self.d(F.silu(self.g(x)) * self.u(x)))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n1 = nn.LayerNorm(cfg.dim)
        self.sa = CausalSA(cfg)
        self.nc = nn.LayerNorm(cfg.dim)
        self.ca = CrossAttn(cfg)
        self.n2 = nn.LayerNorm(cfg.dim)
        self.ff = SwiGLU(cfg)

    def forward(self, x, ctx=None):
        x = x + self.sa(self.n1(x))
        if ctx is not None: x = x + self.ca(self.nc(x), ctx)
        return x + self.ff(self.n2(x))


class TinyGPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.emb    = nn.Embedding(cfg.vocab_size, cfg.dim)
        self.pos    = nn.Embedding(cfg.max_seq_len, cfg.dim)
        self.drop   = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.depth)])
        self.norm   = nn.LayerNorm(cfg.dim)
        self.head   = nn.Linear(cfg.dim, cfg.vocab_size, bias=False)
        self.head.weight = self.emb.weight   # weight tying
        self._init()

    def _init(self):
        for m in self.modules():
            if isinstance(m, (nn.Linear, BitLinear)):
                nn.init.normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding): nn.init.normal_(m.weight, std=0.02)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, ids, ctx=None):
        B,T = ids.shape
        x = self.drop(self.emb(ids) + self.pos(torch.arange(T, device=ids.device)))
        for blk in self.blocks: x = blk(x, ctx)
        return self.head(self.norm(x))


# ==============================================================================
# 9  FULL MODEL
# ==============================================================================

class TinyVLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.proj = Projector(CLIP_DIM, cfg.dim)
        self.lm   = TinyGPT(cfg)

    def encode(self, clip_feat):
        """clip_feat: (B, 196, 768) -> img_feat: (B, 196, dim)"""
        return self.proj(clip_feat.to(device))

    def forward(self, clip_feat, inp_ids, labels):
        img_feat = self.encode(clip_feat)
        logits   = self.lm(inp_ids, img_feat)
        B,T,V    = logits.shape
        return F.cross_entropy(logits.reshape(B*T,V), labels.reshape(B*T),
                               ignore_index=PAD_ID)

    @torch.no_grad()
    def generate(self, clip_feat, max_new=55, temperature=0.7, top_k=50, top_p=0.9):
        self.eval()
        img_feat = self.encode(clip_feat.unsqueeze(0) if clip_feat.dim()==2 else clip_feat)
        ids = torch.tensor([[BOS_ID]], device=device)
        for _ in range(max_new):
            if ids.size(1) >= lcfg.max_seq_len: break
            logits = self.lm(ids, img_feat)[:,-1,:] / temperature
            if top_k > 0:
                thr = torch.topk(logits, top_k).values[:,-1:]
                logits[logits < thr] = -float("inf")
            probs = F.softmax(logits, -1)
            sp, si = torch.sort(probs, descending=True)
            sp[sp.cumsum(-1) - sp > top_p] = 0.0
            sp = sp / sp.sum()
            next_id = si.gather(-1, torch.multinomial(sp, 1))
            if next_id.item() == EOS_ID: break
            ids = torch.cat([ids, next_id], dim=1)
        return decode(ids[0,1:].tolist())

    @torch.no_grad()
    def beam(self, clip_feat, width=5, max_new=55, len_pen=0.7):
        self.eval()
        img_feat = self.encode(clip_feat.unsqueeze(0) if clip_feat.dim()==2 else clip_feat)
        beams: List[Tuple[float,List[int]]] = [(0.0,[BOS_ID])]
        done:  List[Tuple[float,List[int]]] = []
        for _ in range(max_new):
            cands = []
            for score, seq in beams:
                if seq[-1] == EOS_ID: done.append((score,seq)); continue
                ids  = torch.tensor([seq], device=device)
                logp = F.log_softmax(self.lm(ids, img_feat)[:,-1,:], -1)
                tv, ti = torch.topk(logp, width)
                for v,i in zip(tv[0].tolist(), ti[0].tolist()):
                    cands.append((score+v, seq+[i]))
            if not cands: break
            cands.sort(key=lambda x: x[0]/(len(x[1])**len_pen), reverse=True)
            beams = cands[:width]
        done += beams
        best = max(done, key=lambda x: x[0]/(len(x[1])**len_pen))
        return decode(best[1][1:])

    def n_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


model = TinyVLM(lcfg).to(device)
print(f"\nTinyVLM v3  ->  {model.n_params()/1e6:.1f} M trainable parameters")
print(f"  Projector : {sum(p.numel() for p in model.proj.parameters())/1e6:.2f} M")
print(f"  TinyGPT   : {sum(p.numel() for p in model.lm.parameters())/1e6:.2f} M")
bitnet_audit = audit_bitnet_tinygpt(model)
bitnet_pack_probe = check_bitnet_pack_roundtrip(model, max_modules=8)
bitnet_kernel_probe = check_bitnet_kernel_equivalence(model)
bitnet_stats = bitnet_memory_report(model)

# forward sanity check
with torch.no_grad():
    _f = torch.randn(4,196,768)
    _i = torch.randint(0,lcfg.vocab_size,(4,40))
    _l = torch.randint(0,lcfg.vocab_size,(4,40))
    print(f"  Forward OK  loss={model(_f,_i.to(device),_l.to(device)).item():.4f}")
    bitnet_forward_probe = profile_forward_once(model, _f, _i.to(device), _l.to(device), repeats=4, warmup=1)
    if device.type == "cuda":
        bitnet_path_probe = profile_bitnet_paths(model, _f, _i.to(device), _l.to(device), repeats=2, warmup=1)
del _f,_i,_l

  BitNet CUDA kernel compiled

TinyVLM v3  ->  264.1 M trainable parameters
  Projector : 4.46 M
  TinyGPT   : 259.61 M
  BitNet GPT audit OK: 96 BitLinear modules; head tied to embedding
  2-bit pack roundtrip OK: 8 modules
  CUDA equivalence OK: 4 modules, max_err=5.96e-07

BitNet report
  BitLinear modules       : 96
  BitLinear weights       : 251.66 M
  Ternary -1/0/+1        : 34.5%/31.0%/34.5%
  Shadow model            : 1007.3 MiB
  GPT BitLinear packed 2b : 60.00 MiB
  GPT theoretical 1.58b   : 47.55 MiB
  Process RAM             : 5.55 GiB
  VRAM alloc/reserved     : 1.00/1.12 GiB
  Forward OK  loss=9.1560
  Forward benchmark: 345.63 ms  RAM=5.56 GiB  peakVRAM=1.09 GiB
  Forward benchmark: 111.91 ms  RAM=5.56 GiB  peakVRAM=1.12 GiB
  Forward benchmark: 347.72 ms  RAM=5.56 GiB  peakVRAM=1.07 GiB
  BitNet CUDA/PyTorch speedup: 0.32x


## 7. Dataset, DataLoader and Optimizer

In [ ]:
# ==============================================================================
# 10  DATASET — reads features from cache, does not load CLIP
# ==============================================================================

class CachedDS(Dataset):
    """
    Reads features directly from RAM (feat_cache).
    __getitem__ is effectively instant — zero disk I/O.
    """
    def __init__(self, pairs, max_len):
        self.max_len = max_len
        self.items   = []   # (img_path, caption)
        for path, cap in pairs:
            if Path(path).stem in feat_cache:
                self.items.append((path, cap))
        print(f"  Dataset: {len(self.items):,} pairs in RAM")

    def __len__(self): return len(self.items)

    def __getitem__(self, idx):
        path, cap = self.items[idx]
        feat = feat_cache[Path(path).stem]           # (196, 768) fp32 from RAM
        toks = encode(cap)
        toks += [PAD_ID] * (self.max_len - len(toks))
        toks  = toks[:self.max_len]
        inp   = torch.tensor(toks[:-1], dtype=torch.long)
        lbl   = torch.tensor(toks[1:],  dtype=torch.long)
        return {"feat": feat, "inp": inp, "lbl": lbl, "path": path, "cap": cap}


def collate(batch):
    batch = [b for b in batch if b is not None]
    if not batch: return None
    return {
        "feat": torch.stack([b["feat"] for b in batch]),
        "inp":  torch.stack([b["inp"]  for b in batch]),
        "lbl":  torch.stack([b["lbl"]  for b in batch]),
        "path": [b["path"] for b in batch],
        "cap":  [b["cap"]  for b in batch],
    }


print("Building datasets ...")
train_ds = CachedDS(train_pairs, lcfg.max_seq_len)
val_ds   = CachedDS(val_pairs,   lcfg.max_seq_len)

train_dl = DataLoader(train_ds, batch_size=tcfg.batch_size, shuffle=True,
                      num_workers=2, pin_memory=True, collate_fn=collate,
                      drop_last=True)
val_dl   = DataLoader(val_ds, batch_size=tcfg.batch_size, shuffle=False,
                      num_workers=2, pin_memory=True, collate_fn=collate)

print(f"train={len(train_dl)} batches  val={len(val_dl)} batches")


# ==============================================================================
# 11  OPTIMIZER & SCHEDULER
# ==============================================================================

def build_opt(model):
    skip  = ("bias","norm","emb","pos","skip")
    decay, nodecay = [], []
    for n,p in model.named_parameters():
        if not p.requires_grad: continue
        (nodecay if any(k in n for k in skip) else decay).append(p)
    try:
        opt = torch.optim.AdamW(
            [{"params":decay,"weight_decay":tcfg.weight_decay},
             {"params":nodecay,"weight_decay":0.0}],
            lr=tcfg.lr, betas=(tcfg.beta1,tcfg.beta2), fused=True)
    except TypeError:
        opt = torch.optim.AdamW(
            [{"params":decay,"weight_decay":tcfg.weight_decay},
             {"params":nodecay,"weight_decay":0.0}],
            lr=tcfg.lr, betas=(tcfg.beta1,tcfg.beta2))
    return opt


def cosine_lr(step):
    if step < tcfg.warmup_steps:
        return tcfg.lr * step / max(tcfg.warmup_steps,1)
    if step >= tcfg.max_steps: return tcfg.min_lr
    t = (step - tcfg.warmup_steps)/(tcfg.max_steps - tcfg.warmup_steps)
    return tcfg.min_lr + 0.5*(tcfg.lr-tcfg.min_lr)*(1+math.cos(math.pi*t))


optimizer = build_opt(model)
scaler    = torch.cuda.amp.GradScaler(enabled=tcfg.mixed_precision)
print("Optimizer ready")

Building datasets ...
  Dataset: 22,512 pairs in RAM
  Dataset: 2,502 pairs in RAM
train=351 batches  val=40 batches
Optimizer ready


/tmp/ipykernel_17639/3807156785.py:88: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler(enabled=tcfg.mixed_precision)


## 8. Training


In [ ]:
# ==============================================================================
# 12  TRAINING
# ==============================================================================

@torch.no_grad()
def evaluate(max_b=50):
    model.eval()
    tot, n = 0.0, 0
    for b in val_dl:
        if n >= max_b or b is None: break
        feat = b["feat"].to(device, non_blocking=True)
        inp  = b["inp"].to(device,  non_blocking=True)
        lbl  = b["lbl"].to(device,  non_blocking=True)
        with torch.cuda.amp.autocast(enabled=tcfg.mixed_precision, dtype=DTYPE):
            tot += model(feat, inp, lbl).item()
        n += 1
    model.train()
    return tot / max(n,1)


def save(step, loss):
    p = tcfg.root/"ckpt"/f"step{step:06d}.pt"
    torch.save({"step":step,"model":model.state_dict(),
                "opt":optimizer.state_dict(),"loss":loss}, p)
    tqdm.write(f"   {p.name}  (loss={loss:.4f})")


history = {"step":[],"train":[],"vstep":[],"val":[],"rss_gb":[],"vram_gb":[],"step_s":[]}
model.train(); optimizer.zero_grad(set_to_none=True)
if device.type == "cuda": torch.cuda.reset_peak_memory_stats()
accum, t0 = 0.0, time.time()
pbar = tqdm(range(tcfg.max_steps), desc="train", unit="step")

print("="*55)
print("  TinyVLM v3 — cached features, CLIP not used in the loop")
print("  Loss should drop quickly (<2.5 within ~500 steps)")
print("="*55)

_it = iter(train_dl)
for step in pbar:
    for pg in optimizer.param_groups: pg["lr"] = cosine_lr(step)

    try: batch = next(_it)
    except StopIteration: _it = iter(train_dl); batch = next(_it)
    if batch is None: continue

    feat = batch["feat"].to(device, non_blocking=True)
    inp  = batch["inp"].to(device,  non_blocking=True)
    lbl  = batch["lbl"].to(device,  non_blocking=True)

    with torch.cuda.amp.autocast(enabled=tcfg.mixed_precision, dtype=DTYPE):
        loss = model(feat, inp, lbl)

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    nn.utils.clip_grad_norm_(model.parameters(), tcfg.grad_clip)
    scaler.step(optimizer); scaler.update()
    optimizer.zero_grad(set_to_none=True)
    accum += loss.item()

    if step % tcfg.log_every == 0 and step > 0:
        tl    = accum / tcfg.log_every; accum = 0.0
        ppl   = math.exp(min(tl,20))
        elapsed = time.time()-t0
        sps   = step / elapsed
        rss   = rss_gb(); vram = cuda_mem_gb("allocated")
        step_s = elapsed / max(step,1)
        history["step"].append(step); history["train"].append(tl)
        history["rss_gb"].append(rss); history["vram_gb"].append(vram); history["step_s"].append(step_s)
        pbar.set_postfix(loss=f"{tl:.4f}", ppl=f"{ppl:.1f}",
                         lr=f"{cosine_lr(step):.2e}", sps=f"{sps:.1f}",
                         ram=f"{rss:.1f}G", vram=f"{vram:.1f}G")

    if step % tcfg.eval_every == 0 and step > 0:
        vl = evaluate()
        history["vstep"].append(step); history["val"].append(vl)
        tqdm.write(f"  [step {step:5d}]  val_loss={vl:.4f}  ppl={math.exp(min(vl,20)):.1f}")

    if step % tcfg.save_every == 0 and step > 0:
        save(step, history["train"][-1] if history["train"] else 0.0)

save(step, history["train"][-1] if history["train"] else 0.0)
print(f"\nTraining complete in {(time.time()-t0)/60:.1f} min")
print(f"  RAM final={rss_gb():.2f} GiB  peakVRAM={cuda_peak_gb():.2f} GiB")
packed_path = tcfg.root/"ckpt"/"tinygpt_bitnet_packed_2bit.pt"
export_bitnet_packed_gpt(model, packed_path)
print(f"  Packed GPT size={packed_path.stat().st_size/1024**2:.2f} MiB")
bitnet_stats_final = bitnet_memory_report(model)

train:   0%|          | 0/8000 [00:00<?, ?step/s]

  TinyVLM v3 — cached features, CLIP not used in the loop
  Loss should drop quickly (<2.5 within ~500 steps)


/tmp/ipykernel_17639/4252014902.py:51: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=tcfg.mixed_precision, dtype=DTYPE):
train:   5%|▌         | 400/8000 [05:04<1:37:29,  1.30step/s, loss=3.0259, lr=3.00e-04, ppl=20.6, ram=5.9G, sps=1.3, vram=3.1G]/tmp/ipykernel_17639/4252014902.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=tcfg.mixed_precision, dtype=DTYPE):
train:   5%|▌         | 401/8000 [10:21<202:22:09, 95.87s/step, loss=3.0259, lr=3.00e-04, ppl=20.6, ram=5.9G, sps=1.3, vram=3.1G]

  [step   400]  val_loss=7.7091  ppl=2228.6


train:  10%|█         | 801/8000 [20:44<191:41:55, 95.86s/step, loss=2.5740, lr=2.97e-04, ppl=13.1, ram=6.2G, sps=0.9, vram=3.1G]

...

  [step  7200]  val_loss=8.2355  ppl=3772.3


train:  95%|█████████▌| 7601/8000 [3:24:15<10:37:01, 95.79s/step, loss=0.1978, lr=3.18e-05, ppl=1.2, ram=6.2G, sps=0.6, vram=3.1G]

  [step  7600]  val_loss=8.0843  ppl=3243.2


train: 100%|██████████| 8000/8000 [3:29:20<00:00,  1.57s/step, loss=0.1933, lr=3.00e-05, ppl=1.2, ram=6.2G, sps=0.6, vram=3.1G]


   step007999.pt  (loss=0.1933)

Training complete in 210.9 min
  RAM final=6.20 GiB  peakVRAM=8.15 GiB
  BitNet GPT packed weights saved to /content/tinyvlm3/ckpt/tinygpt_bitnet_packed_2bit.pt
  Packed GPT size=60.12 MiB

BitNet report
  BitLinear modules       : 96
  BitLinear weights       : 251.66 M
  Ternary -1/0/+1        : 34.5%/31.0%/34.5%
  Shadow model            : 1007.3 MiB
  GPT BitLinear packed 2b : 60.00 MiB
  GPT theoretical 1.58b   : 47.55 MiB
  Process RAM             : 6.20 GiB
  VRAM alloc/reserved     : 3.07/8.54 GiB
